In [1]:
import os
import pandas as pd
import numpy as np

# Step 1: Define CLUSTERS and paths
CLUSTERS = {
    1: ["Bella_Flor", "Filadelfia", "Porvenir", "Puerto_Gonzalo_Moreno",
        "Puerto_Rico", "San_Lorenzo", "Sena", "Villa_Nueva"],
    2: ["Exaltación", "Ingavi", "Nueva_Esperanza", "San_Pedro",
        "Santa_Rosa_Pando", "Santos_Mercado"],
    3: ["Bolpebra"],
    4: ["Guayaramerín", "Riberalta"],
    5: ["Cobija"],
    6: ["Reyes", "Santa_Rosa_Beni", "Ixiamas"],
}

DIR_SOLAR = "data solar"
DIR_WIND = "data wind"
DIR_OUT = "output"

os.makedirs(DIR_OUT, exist_ok=True)

# Helper function to handle naming variations
def find_file(directory, prefix, muni):
    variations = [
        muni,
        muni.replace('_', ' '),
        muni.replace('_', '-'),
        muni.lower(),
        muni.lower().replace('_', ' ')
    ]
    for v in variations:
        path = os.path.join(directory, f"{prefix}_{v}.csv")
        if os.path.exists(path): return path
    return None

# Step 2: Read PV files
def read_pv(filepath):
    df = pd.read_csv(filepath, comment='#')
    df['PV'] = df['electricity']
    df['SOLAR_raw'] = df['irradiance_direct'] + df['irradiance_diffuse']
    return df[['PV', 'SOLAR_raw']]

# Step 3: Read Wind files
def read_wind(filepath):
    df = pd.read_csv(filepath, comment='#')
    df['WIND_ONSHORE'] = df['electricity']
    return df[['WIND_ONSHORE']]

# Step 4: Process clusters
summary_stats = []

for c_id, munis in CLUSTERS.items():
    pv_profiles = []
    solar_raw_profiles = []
    wind_profiles = []
    
    for muni in munis:
        pv_file = find_file(DIR_SOLAR, "ninja_pv", muni)
        wind_file = find_file(DIR_WIND, "ninja_wind", muni)
        
        # Parse PV
        if not pv_file:
            print(f"WARNING: PV file not found for {muni}")
        else:
            df_pv = read_pv(pv_file)
            pv_profiles.append(df_pv['PV'])
            solar_raw_profiles.append(df_pv['SOLAR_raw'])
            
        # Parse Wind
        if not wind_file:
            print(f"WARNING: Wind file not found for {muni}")
        else:
            df_wind = read_wind(wind_file)
            wind_profiles.append(df_wind['WIND_ONSHORE'])
            
    # Aggregate and average per cluster
    df_out = pd.DataFrame()
    
    if pv_profiles:
        # Average PV
        df_out['PV'] = pd.concat(pv_profiles, axis=1).mean(axis=1)
        
        # Calculate SOLAR: average irradiance then normalize by the cluster's absolute max
        solar_raw_avg = pd.concat(solar_raw_profiles, axis=1).mean(axis=1)
        cluster_max_irradiance = pd.concat(solar_raw_profiles, axis=1).max().max() 
        df_out['SOLAR'] = solar_raw_avg / cluster_max_irradiance
    else:
        df_out['PV'] = np.nan
        df_out['SOLAR'] = np.nan
        
    if wind_profiles:
        # Average WIND
        df_out['WIND_ONSHORE'] = pd.concat(wind_profiles, axis=1).mean(axis=1)
    else:
        df_out['WIND_ONSHORE'] = np.nan
        
    # Ensure column order
    df_out = df_out[['PV', 'SOLAR', 'WIND_ONSHORE']]
    
    # Save formatted CSV
    out_path = os.path.join(DIR_OUT, f"renewables_C{c_id}.csv")
    df_out.to_csv(out_path, index=False, float_format="%.6f")
    
    # Collect summary data
    summary_stats.append({
        "Cluster": f"C{c_id}",
        "N_munis": len(munis),
        "PV_mean": df_out['PV'].mean(),
        "PV_max": df_out['PV'].max(),
        "WIND_mean": df_out['WIND_ONSHORE'].mean(),
        "WIND_max": df_out['WIND_ONSHORE'].max(),
        "SOLAR_mean": df_out['SOLAR'].mean()
    })

# Step 5: Print summary table
print("\nProcessing Complete! Summary Table:")
print(f"| {'Cluster':<7} | {'N_munis':<7} | {'PV_mean':<8} | {'PV_max':<8} | {'WIND_mean':<9} | {'WIND_max':<8} | {'SOLAR_mean':<10} |")
print("|" + "-"*9 + "|" + "-"*9 + "|" + "-"*10 + "|" + "-"*10 + "|" + "-"*11 + "|" + "-"*10 + "|" + "-"*12 + "|")
for s in summary_stats:
    print(f"| {s['Cluster']:<7} | {s['N_munis']:<7} | {s['PV_mean']:<8.4f} | {s['PV_max']:<8.4f} | {s['WIND_mean']:<9.4f} | {s['WIND_max']:<8.4f} | {s['SOLAR_mean']:<10.4f} |")


Processing Complete! Summary Table:
| Cluster | N_munis | PV_mean  | PV_max   | WIND_mean | WIND_max | SOLAR_mean |
|---------|---------|----------|----------|-----------|----------|------------|
| C1      | 8       | 0.1739   | 0.7710   | 0.0584    | 0.5619   | 0.2159     |
| C2      | 6       | 0.1710   | 0.7573   | 0.0546    | 0.4808   | 0.2033     |
| C3      | 1       | 0.1748   | 0.7710   | 0.0604    | 0.5740   | 0.2204     |
| C4      | 2       | 0.1645   | 0.7485   | 0.0486    | 0.4305   | 0.2003     |
| C5      | 1       | 0.1752   | 0.7690   | 0.0611    | 0.6740   | 0.2213     |
| C6      | 3       | 0.1615   | 0.7817   | 0.1047    | 0.6967   | 0.1906     |
